# SkyDet on SkySeaLand

Reference implementation for the SkyDet development pipeline on the four-class SkySeaLand satellite dataset.

- Paper: [arXiv:2608.07382](https://arxiv.org/abs/2608.07382)
- Dataset: [Mendeley Data](https://doi.org/10.17632/d42n3cp86p.3) and [Kaggle](https://www.kaggle.com/datasets/mdzahidhasanriad/skysealand-coco)
- Classes: `airplane`, `boat`, `car`, `ship`

> **Artifact note:** this notebook preserves a 100-epoch development configuration. The manuscript reports the finalized 150-epoch reference run. See [`docs/REPRODUCIBILITY.md`](../docs/REPRODUCIBILITY.md) before comparing metrics.


In [ ]:
# SkyDet
# MobileNetV3 Small backbone, PANet neck with optional GhostConv, FCOS head
# COCO format dataset, Kaggle and Colab friendly
# Includes data audit, visual reports, training, evaluation, fp16 smaller checkpoint, and industry style inference engine

import os
import json
import time
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
import albumentations as A

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

import torchvision
from torchvision.models.feature_extraction import create_feature_extractor

## Configuration

Central experiment, model, data, and evaluation settings.

In [ ]:
from dataclasses import dataclass
from typing import Tuple

@dataclass
class CFG:
    # Repro
    seed: int = 42

    # Device
    device: str = "cuda"
    amp: bool = True
    compile_model: bool = False

    # Paths
    data_root: str = "/kaggle/input/datasets/mdzahidhasanriad/skysealand-coco"
    train_images: str = "train"
    val_images: str = "valid"
    test_images: str = "test"

    ann_train: str = "train/_annotations.coco.json"
    ann_val: str = "valid/_annotations.coco.json"
    ann_test: str = "test/_annotations.coco.json"

    # Classes
    class_names: Tuple[str, ...] = (
        "airplane",
        "boat",
        "car",
        "ship"
    )

    num_classes: int = 4

    # Image and loader
    image_size: int = 640
    batch_size: int = 8
    num_workers: int = 2
    pin_memory: bool = True
    persistent_workers: bool = True

    # Training
    epochs: int = 100
    lr_head: float = 3e-4
    lr_backbone: float = 1e-4
    weight_decay: float = 0.05
    grad_clip_norm: float = 0.5

    # Model name and output
    model_name: str = "SkyDet"
    out_dir: str = "/kaggle/working/outputs_Riad/outputs_SkyDet"

    # Lightweight knobs for size and speed
    fpn_dim: int = 64

    # Head knobs
    head_depth: int = 2
    use_depthwise_head: bool = True

    # Backbone
    use_pretrained_backbone: bool = True
    freeze_backbone_epochs: int = 1

    # Small object boost
    use_stride4: bool = True

    # Ghost in neck
    use_ghost_neck: bool = True
    ghost_ratio: int = 2

    # Optional attention
    use_se_neck: bool = True
    use_cbam_neck: bool = True

    # Optional dilation
    use_dilated_conv: bool = True
    dilation_rate: int = 2

    # Loss weights
    loss_weight_cls: float = 1.0
    loss_weight_reg: float = 1.0
    loss_weight_ctr: float = 1.0

    # FCOS level ranges in resized image space
    range_s4: Tuple[float, float] = (0.0, 64.0)
    range_s8: Tuple[float, float] = (64.0, 128.0)
    range_s16: Tuple[float, float] = (128.0, 256.0)
    range_s32: Tuple[float, float] = (256.0, 1e8)

    # Eval
    eval_every: int = 1
    eval_force_topk: int = 50
    eval_thr_early: float = 0.001
    eval_thr_mid: float = 0.01
    eval_thr_late: float = 0.05
    eval_thr_mid_epoch: int = 10
    eval_thr_late_epoch: int = 20

    # Inference
    viz_score_thresh: float = 0.25
    nms_thresh: float = 0.45
    max_det: int = 300

    # Save smaller checkpoint
    save_fp16_checkpoint: bool = True

    # Optional ONNX export
    export_onnx: bool = False
    onnx_path: str = "/kaggle/working/outputs_Riad/SkyDetHybrid.onnx"

    # Visual audit outputs
    audit_num_overlay: int = 24
    audit_num_badbox_overlay: int = 24


cfg = CFG()

## Core utilities

Reproducibility, geometry conversion, letterboxing, and visualization helpers.

In [ ]:
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def now_str():
    return time.strftime("%Y%m%d_%H%M%S")

def to_xyxy_from_coco_bbox(b: List[float]) -> List[float]:
    x, y, w, h = b
    return [x, y, x + w, y + h]

def letterbox_resize(img: np.ndarray, boxes_xyxy: Optional[np.ndarray], out_size: int):
    h0, w0 = img.shape[:2]
    s = out_size / max(h0, w0)
    nh = int(round(h0 * s))
    nw = int(round(w0 * s))

    img_rs = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)

    pad_h = out_size - nh
    pad_w = out_size - nw
    top = pad_h // 2
    left = pad_w // 2

    canvas = np.zeros((out_size, out_size, 3), dtype=img_rs.dtype)
    canvas[top:top + nh, left:left + nw] = img_rs

    if boxes_xyxy is not None and boxes_xyxy.shape[0] > 0:
        b = boxes_xyxy.copy()
        b[:, 0] = b[:, 0] * s + left
        b[:, 1] = b[:, 1] * s + top
        b[:, 2] = b[:, 2] * s + left
        b[:, 3] = b[:, 3] * s + top
        boxes_xyxy = b

    meta = {
        "orig_h": int(h0),
        "orig_w": int(w0),
        "scale": float(s),
        "pad_left": int(left),
        "pad_top": int(top),
    }
    return canvas, boxes_xyxy, meta

def clamp_xyxy(boxes: np.ndarray, w: int, h: int):
    b = boxes.copy()
    b[:, 0] = np.clip(b[:, 0], 0, w - 1)
    b[:, 2] = np.clip(b[:, 2], 0, w - 1)
    b[:, 1] = np.clip(b[:, 1], 0, h - 1)
    b[:, 3] = np.clip(b[:, 3], 0, h - 1)
    return b

def xywh_from_xyxy(box: np.ndarray):
    x1, y1, x2, y2 = box.tolist()
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]

def draw_boxes(img_rgb: np.ndarray, boxes_xyxy: List[List[float]], labels: List[str], scores: Optional[List[float]] = None, color=(0, 255, 0)):
    im = img_rgb.copy()
    for i, b in enumerate(boxes_xyxy):
        x1, y1, x2, y2 = [int(round(v)) for v in b]
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(im.shape[1] - 1, x2)
        y2 = min(im.shape[0] - 1, y2)
        cv2.rectangle(im, (x1, y1), (x2, y2), color, 2)

        text = labels[i]
        if scores is not None:
            text = f"{text} {scores[i]:.2f}"
        y_text = max(0, y1 - 5)
        cv2.putText(im, text, (x1, y_text), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return im

def get_eval_score_thresh(epoch_1based: int) -> float:
    if epoch_1based < cfg.eval_thr_mid_epoch:
        return cfg.eval_thr_early
    if epoch_1based < cfg.eval_thr_late_epoch:
        return cfg.eval_thr_mid
    return cfg.eval_thr_late

## Dataset audit

COCO validation, distribution reports, and ground-truth overlays.

In [ ]:
def load_coco_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def audit_coco_split(ann_path: str, images_dir: str, keep_cat_ids: set):
    coco = load_coco_json(ann_path)
    id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}

    missing_files = []
    for img_id, fn in id_to_file.items():
        fp = os.path.join(images_dir, fn)
        if not os.path.exists(fp):
            missing_files.append((int(img_id), fn))

    bad_boxes = []
    per_class = {}
    areas = []

    for ann in coco["annotations"]:
        if ann.get("iscrowd", 0) == 1:
            continue
        cid = int(ann["category_id"])
        if cid not in keep_cat_ids:
            continue

        per_class[cid] = per_class.get(cid, 0) + 1
        x, y, w, h = ann["bbox"]

        if w <= 1 or h <= 1:
            bad_boxes.append((int(ann["image_id"]), int(cid), [float(x), float(y), float(w), float(h)], "tiny_wh"))
            continue

        areas.append(float(w * h))

    if len(areas) == 0:
        areas = [0.0]

    out = {
        "num_images": int(len(id_to_file)),
        "missing_images": int(len(missing_files)),
        "missing_list": missing_files,
        "bad_boxes": int(len(bad_boxes)),
        "bad_boxes_list": bad_boxes,
        "per_class_instances": {int(k): int(v) for k, v in per_class.items()},
        "area_p10": float(np.percentile(np.array(areas, dtype=np.float32), 10)),
        "area_p50": float(np.percentile(np.array(areas, dtype=np.float32), 50)),
        "area_p90": float(np.percentile(np.array(areas, dtype=np.float32), 90)),
    }
    return out

def save_audit_reports(audit: Dict[str, Any], out_dir: str, split_name: str):
    ensure_dir(out_dir)

    json_path = os.path.join(out_dir, f"audit_{split_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2)

    missing_txt = os.path.join(out_dir, f"missing_images_{split_name}.txt")
    with open(missing_txt, "w", encoding="utf-8") as f:
        for img_id, fn in audit["missing_list"]:
            f.write(f"{img_id}\t{fn}\n")

    bad_txt = os.path.join(out_dir, f"bad_boxes_{split_name}.txt")
    with open(bad_txt, "w", encoding="utf-8") as f:
        for img_id, cid, bb, reason in audit["bad_boxes_list"]:
            f.write(f"{img_id}\t{cid}\t{reason}\t{bb}\n")

    # Plot charts when matplotlib is available
    try:
        import matplotlib.pyplot as plt

        # per class bar
        if len(audit["per_class_instances"]) > 0:
            keys = list(audit["per_class_instances"].keys())
            vals = [audit["per_class_instances"][k] for k in keys]
            plt.figure()
            plt.bar([str(k) for k in keys], vals)
            plt.xlabel("category_id")
            plt.ylabel("instances")
            plt.title(f"Per class instances {split_name}")
            plt.savefig(os.path.join(out_dir, f"per_class_instances_{split_name}.png"), dpi=160, bbox_inches="tight")
            plt.close()

        # area histogram
        # We do not store full areas list in audit, we rebuild from json quickly
        # If you want full area hist, re parse annotations here
        plt.figure()
        plt.bar(["p10", "p50", "p90"], [audit["area_p10"], audit["area_p50"], audit["area_p90"]])
        plt.ylabel("bbox area")
        plt.title(f"Area percentiles {split_name}")
        plt.savefig(os.path.join(out_dir, f"bbox_area_percentiles_{split_name}.png"), dpi=160, bbox_inches="tight")
        plt.close()

    except Exception as e:
        print("Audit plotting skipped:", e)

def overlay_random_gt_samples(images_dir: str, ann_path: str, out_dir: str, keep_cat_ids: set, num_samples: int = 16):
    ensure_dir(out_dir)
    coco = load_coco_json(ann_path)

    id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}
    id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    anns_by_img: Dict[int, List[Dict[str, Any]]] = {}
    for ann in coco["annotations"]:
        if ann.get("iscrowd", 0) == 1:
            continue
        img_id = int(ann["image_id"])
        anns_by_img.setdefault(img_id, []).append(ann)

    all_ids = list(id_to_file.keys())
    random.shuffle(all_ids)
    picked = 0

    for img_id in all_ids:
        fn = id_to_file[img_id]
        fp = os.path.join(images_dir, fn)
        if not os.path.exists(fp):
            continue

        img = cv2.imread(fp, cv2.IMREAD_COLOR)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        boxes = []
        labels = []
        for ann in anns_by_img.get(int(img_id), []):
            cid = int(ann["category_id"])
            if cid not in keep_cat_ids:
                continue
            x, y, bw, bh = ann["bbox"]

            # Handle normalized bboxes
            if max(x, y, bw, bh) <= 1.5:
                x = x * w
                bw = bw * w
                y = y * h
                bh = bh * h

            if bw <= 1 or bh <= 1:
                continue

            xyxy = [x, y, x + bw, y + bh]
            boxes.append([float(v) for v in xyxy])
            labels.append(id_to_name.get(cid, str(cid)))

        if len(boxes) == 0:
            continue

        vis = draw_boxes(img, boxes, labels, scores=None, color=(0, 255, 0))
        out_path = os.path.join(out_dir, f"gt_{int(img_id)}_{os.path.splitext(fn)[0]}.jpg")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        picked += 1
        if picked >= num_samples:
            break

def overlay_bad_box_samples(images_dir: str, ann_path: str, out_dir: str, keep_cat_ids: set, num_samples: int = 16):
    ensure_dir(out_dir)
    coco = load_coco_json(ann_path)
    id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}
    id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

    # Collect bad boxes
    bad_list = []
    for ann in coco["annotations"]:
        if ann.get("iscrowd", 0) == 1:
            continue
        cid = int(ann["category_id"])
        if cid not in keep_cat_ids:
            continue
        x, y, bw, bh = ann["bbox"]
        if bw <= 1 or bh <= 1:
            bad_list.append((int(ann["image_id"]), cid, [x, y, bw, bh]))

    random.shuffle(bad_list)
    saved = 0
    for img_id, cid, bb in bad_list:
        fn = id_to_file.get(int(img_id), None)
        if fn is None:
            continue
        fp = os.path.join(images_dir, fn)
        if not os.path.exists(fp):
            continue

        img = cv2.imread(fp, cv2.IMREAD_COLOR)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        x, y, bw, bh = bb
        if max(x, y, bw, bh) <= 1.5:
            x = x * w
            bw = bw * w
            y = y * h
            bh = bh * h

        xyxy = [x, y, x + bw, y + bh]
        label = id_to_name.get(int(cid), str(cid))
        vis = draw_boxes(img, [xyxy], [f"bad {label}"], scores=None, color=(0, 0, 255))

        out_path = os.path.join(out_dir, f"bad_{int(img_id)}_{os.path.splitext(fn)[0]}.jpg")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        saved += 1
        if saved >= num_samples:
            break

## Data pipeline

COCO loading, augmentation, tensor conversion, and batching.

In [ ]:
class CocoDetDataset(Dataset):
    def __init__(self, images_dir: str, ann_path: str, image_size: int, is_train: bool, cat_id_to_contig: Dict[int, int]):
        self.images_dir = images_dir
        self.ann_path = ann_path
        self.image_size = image_size
        self.is_train = is_train
        self.cat_id_to_contig = cat_id_to_contig

        coco = load_coco_json(ann_path)
        self.imgs = {int(img["id"]): img for img in coco["images"]}

        self.anns_by_img: Dict[int, List[Dict[str, Any]]] = {}
        for ann in coco["annotations"]:
            if ann.get("iscrowd", 0) == 1:
                continue
            img_id = int(ann["image_id"])
            self.anns_by_img.setdefault(img_id, []).append(ann)

        ids = []
        for img_id, info in self.imgs.items():
            fp = os.path.join(images_dir, info["file_name"])
            if os.path.exists(fp):
                ids.append(int(img_id))
        self.ids = ids

        self.tf = self._build_tf()

    def _build_tf(self):
        if not self.is_train:
            return None

        # Use resize plus pad, keep bboxes in pascal voc format
        return A.Compose(
            [
                A.LongestMaxSize(max_size=self.image_size),
                A.PadIfNeeded(min_height=self.image_size, min_width=self.image_size, border_mode=0),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.HueSaturationValue(p=0.2),
                A.GaussianBlur(p=0.1),
                A.Affine(scale=(0.85, 1.15), translate_percent=(0.0, 0.08), rotate=(-10, 10), p=0.4),
            ],
            bbox_params=A.BboxParams(
                format="pascal_voc",
                label_fields=["labels"],
                min_visibility=0.2,
                clip=True,
                filter_invalid_bboxes=True,
            ),
        )

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx: int):
        img_id = int(self.ids[idx])
        info = self.imgs[img_id]
        path = os.path.join(self.images_dir, info["file_name"])

        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ih, iw = img.shape[:2]
        anns = self.anns_by_img.get(img_id, [])
        boxes = []
        labels = []

        for a in anns:
            cid = int(a["category_id"])
            if cid not in self.cat_id_to_contig:
                continue

            x, y, bw, bh = a["bbox"]

            # Handle normalized bboxes
            if max(x, y, bw, bh) <= 1.5:
                x = x * iw
                bw = bw * iw
                y = y * ih
                bh = bh * ih

            if bw <= 1 or bh <= 1:
                continue

            x1, y1, x2, y2 = x, y, x + bw, y + bh
            x1 = float(np.clip(x1, 0, iw - 1))
            y1 = float(np.clip(y1, 0, ih - 1))
            x2 = float(np.clip(x2, 0, iw - 1))
            y2 = float(np.clip(y2, 0, ih - 1))

            if (x2 - x1) <= 1 or (y2 - y1) <= 1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_contig[cid])

        if len(boxes) == 0:
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,), dtype=np.int64)
        else:
            boxes = np.array(boxes, dtype=np.float32)
            labels = np.array(labels, dtype=np.int64)

        if self.is_train:
            out = self.tf(image=img, bboxes=boxes.tolist(), labels=labels.tolist())
            img = out["image"]
            boxes = np.array(out["bboxes"], dtype=np.float32)
            labels = np.array(out["labels"], dtype=np.int64)
            meta = {"orig_h": int(img.shape[0]), "orig_w": int(img.shape[1]), "scale": 1.0, "pad_left": 0, "pad_top": 0}
        else:
            img_lb, boxes_lb, meta = letterbox_resize(img, boxes, self.image_size)
            img = img_lb
            boxes = np.zeros((0, 4), dtype=np.float32) if boxes_lb is None else boxes_lb.astype(np.float32)

        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        target = {
            "image_id": int(img_id),
            "boxes": torch.from_numpy(boxes).float(),
            "labels": torch.from_numpy(labels).long(),
            "meta": meta,
        }
        return img, target

def collate_fn(batch):
    imgs, targets = zip(*batch)
    return torch.stack(list(imgs), dim=0), list(targets)

## Detection losses

Class-weighted focal loss and IoU regression loss.

In [ ]:
def sigmoid_focal_loss(logits: torch.Tensor, targets: torch.Tensor, alpha_vec: Optional[torch.Tensor] = None, gamma: float = 2.0) -> torch.Tensor:
    p = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1.0 - p) * (1.0 - targets)
    loss = ce * ((1.0 - p_t) ** gamma)
    if alpha_vec is not None:
        a = alpha_vec.to(logits.device).view(1, -1)
        a_t = a * targets + (1.0 - a) * (1.0 - targets)
        loss = a_t * loss
    return loss

def iou_loss(pred_ltrb: torch.Tensor, tgt_ltrb: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pl, pt, pr, pb = pred_ltrb.unbind(-1)
    tl, tt, tr, tb = tgt_ltrb.unbind(-1)

    p_area = (pl + pr) * (pt + pb)
    t_area = (tl + tr) * (tt + tb)

    iw = torch.min(pl, tl) + torch.min(pr, tr)
    ih = torch.min(pt, tt) + torch.min(pb, tb)
    inter = iw * ih
    union = p_area + t_area - inter
    iou = (inter + eps) / (union + eps)
    return 1.0 - iou

## Model building blocks

Convolution, GhostConv, attention, feature fusion, backbone, and detection head.

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, cin: int, cout: int, k: int, s: int = 1, p: Optional[int] = None, g: int = 1, act: bool = True, dilation: int = 1):
        super().__init__()
        if p is None:
            p = (k // 2) * dilation
        self.conv = nn.Conv2d(cin, cout, k, stride=s, padding=p, groups=g, bias=False, dilation=dilation)
        self.bn = nn.BatchNorm2d(cout)
        self.act = nn.SiLU() if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class GhostConv(nn.Module):
    def __init__(self, cin: int, cout: int, k: int = 1, s: int = 1, ratio: int = 2, dilation: int = 1):
        super().__init__()
        self.cout = cout
        primary = int(math.ceil(cout / ratio))
        cheap = cout - primary

        self.primary = ConvBNAct(cin, primary, k, s=s, p=None, g=1, act=True, dilation=dilation)
        if cheap > 0:
            self.cheap = ConvBNAct(primary, cheap, 3, s=1, p=None, g=primary, act=True, dilation=1)
        else:
            self.cheap = None

    def forward(self, x):
        y = self.primary(x)
        if self.cheap is None:
            return y
        z = self.cheap(y)
        out = torch.cat([y, z], dim=1)
        return out[:, : self.cout, :, :]

def make_neck_conv(cin: int, cout: int, use_ghost: bool, ghost_ratio: int, k: int = 1, s: int = 1, dilation: int = 1):
    if use_ghost:
        return GhostConv(cin, cout, k=k, s=s, ratio=ghost_ratio, dilation=dilation)
    return ConvBNAct(cin, cout, k, s=s, p=None, g=1, act=True, dilation=dilation)

class SqueezeExcite(nn.Module):
    def __init__(self, c: int, r: int = 8):
        super().__init__()
        hidden = max(8, c // r)
        self.fc1 = nn.Conv2d(c, hidden, 1)
        self.fc2 = nn.Conv2d(hidden, c, 1)

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        s = F.silu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s))
        return x * s

class CBAM(nn.Module):
    def __init__(self, c: int, r: int = 8, k: int = 7):
        super().__init__()
        hidden = max(8, c // r)
        self.mlp1 = nn.Conv2d(c, hidden, 1)
        self.mlp2 = nn.Conv2d(hidden, c, 1)
        self.spatial = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        # Channel attention
        avg = F.adaptive_avg_pool2d(x, 1)
        mx = F.adaptive_max_pool2d(x, 1)
        c_att = torch.sigmoid(self.mlp2(F.silu(self.mlp1(avg))) + self.mlp2(F.silu(self.mlp1(mx))))
        x = x * c_att
        # Spatial attention
        avg2 = torch.mean(x, dim=1, keepdim=True)
        mx2, _ = torch.max(x, dim=1, keepdim=True)
        s_att = torch.sigmoid(self.spatial(torch.cat([avg2, mx2], dim=1)))
        return x * s_att

In [ ]:
class PANet4(nn.Module):
    def __init__(self, in_channels: List[int], fpn_dim: int, use_ghost: bool, ghost_ratio: int, use_stride4: bool, use_se: bool, use_cbam: bool, use_dilation: bool, dilation_rate: int):
        super().__init__()
        self.use_stride4 = use_stride4
        self.use_se = use_se
        self.use_cbam = use_cbam

        dilation = dilation_rate if use_dilation else 1

        if self.use_stride4:
            c2, c3, c4, c5 = in_channels
            self.lat2 = make_neck_conv(c2, fpn_dim, use_ghost, ghost_ratio, k=1, s=1, dilation=1)
        else:
            c3, c4, c5 = in_channels

        self.lat3 = make_neck_conv(c3, fpn_dim, use_ghost, ghost_ratio, k=1, s=1, dilation=1)
        self.lat4 = make_neck_conv(c4, fpn_dim, use_ghost, ghost_ratio, k=1, s=1, dilation=1)
        self.lat5 = make_neck_conv(c5, fpn_dim, use_ghost, ghost_ratio, k=1, s=1, dilation=1)

        self.out2 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation) if self.use_stride4 else None
        self.out3 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation)
        self.out4 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation)
        self.out5 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation)

        self.down23 = ConvBNAct(fpn_dim, fpn_dim, 3, s=2, p=None, g=1, act=True) if self.use_stride4 else None
        self.down34 = ConvBNAct(fpn_dim, fpn_dim, 3, s=2, p=None, g=1, act=True)
        self.down45 = ConvBNAct(fpn_dim, fpn_dim, 3, s=2, p=None, g=1, act=True)

        self.pan3 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation) if self.use_stride4 else None
        self.pan4 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation)
        self.pan5 = make_neck_conv(fpn_dim, fpn_dim, use_ghost, ghost_ratio, k=3, s=1, dilation=dilation)

        if self.use_se:
            self.se2 = SqueezeExcite(fpn_dim) if self.use_stride4 else None
            self.se3 = SqueezeExcite(fpn_dim)
            self.se4 = SqueezeExcite(fpn_dim)
            self.se5 = SqueezeExcite(fpn_dim)
        else:
            self.se2 = self.se3 = self.se4 = self.se5 = None

        if self.use_cbam:
            self.cb2 = CBAM(fpn_dim) if self.use_stride4 else None
            self.cb3 = CBAM(fpn_dim)
            self.cb4 = CBAM(fpn_dim)
            self.cb5 = CBAM(fpn_dim)
        else:
            self.cb2 = self.cb3 = self.cb4 = self.cb5 = None

    def _att(self, x, se, cb):
        if se is not None:
            x = se(x)
        if cb is not None:
            x = cb(x)
        return x

    def forward(self, feats: List[torch.Tensor]) -> List[torch.Tensor]:
        if self.use_stride4:
            c2, c3, c4, c5 = feats
            p2 = self.lat2(c2)
        else:
            c3, c4, c5 = feats

        p3 = self.lat3(c3)
        p4 = self.lat4(c4)
        p5 = self.lat5(c5)

        # Top down
        p4 = p4 + F.interpolate(p5, size=p4.shape[-2:], mode="nearest")
        p3 = p3 + F.interpolate(p4, size=p3.shape[-2:], mode="nearest")
        if self.use_stride4:
            p2 = p2 + F.interpolate(p3, size=p2.shape[-2:], mode="nearest")

        # Smooth
        if self.use_stride4:
            p2 = self._att(self.out2(p2), self.se2, self.cb2)
        p3 = self._att(self.out3(p3), self.se3, self.cb3)
        p4 = self._att(self.out4(p4), self.se4, self.cb4)
        p5 = self._att(self.out5(p5), self.se5, self.cb5)

        # Bottom up
        if self.use_stride4:
            n3 = p3 + self.down23(p2)
            n3 = self.pan3(n3)
        else:
            n3 = p3

        n4 = p4 + self.down34(n3)
        n5 = p5 + self.down45(n4)

        n4 = self.pan4(n4)
        n5 = self.pan5(n5)

        if self.use_stride4:
            return [p2, n3, n4, n5]
        return [n3, n4, n5]

In [ ]:
class Scale(nn.Module):
    def __init__(self, init: float = 1.0):
        super().__init__()
        self.s = nn.Parameter(torch.tensor(init, dtype=torch.float32))

    def forward(self, x):
        return x * self.s

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, c: int, dilation: int = 1):
        super().__init__()
        p = dilation
        self.dw = nn.Conv2d(c, c, 3, padding=p, groups=c, bias=False, dilation=dilation)
        self.pw = nn.Conv2d(c, c, 1, bias=False)
        self.gn = nn.GroupNorm(16, c)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = self.gn(x)
        return F.silu(x)

class FCOSHead(nn.Module):
    def __init__(self, fpn_dim: int, num_classes: int, num_levels: int, depth: int, use_depthwise: bool, use_dilation: bool, dilation_rate: int):
        super().__init__()
        dilation = dilation_rate if use_dilation else 1

        def tower():
            layers = []
            for _ in range(depth):
                if use_depthwise:
                    layers.append(DepthwiseSeparableConv(fpn_dim, dilation=dilation))
                else:
                    layers.append(nn.Conv2d(fpn_dim, fpn_dim, 3, padding=dilation, dilation=dilation))
                    layers.append(nn.GroupNorm(16, fpn_dim))
                    layers.append(nn.SiLU())
            return nn.Sequential(*layers)

        self.cls_tower = tower()
        self.reg_tower = tower()

        self.cls_logits = nn.Conv2d(fpn_dim, num_classes, 3, padding=1)
        self.bbox_pred = nn.Conv2d(fpn_dim, 4, 3, padding=1)
        self.ctr_logits = nn.Conv2d(fpn_dim, 1, 3, padding=1)

        self.scales = nn.ModuleList([Scale(1.0) for _ in range(num_levels)])

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        prior_prob = 0.01
        bias = -torch.log(torch.tensor((1.0 - prior_prob) / prior_prob))
        nn.init.constant_(self.cls_logits.bias, bias)

    def forward(self, feats: List[torch.Tensor]):
        cls_out, reg_out, ctr_out = [], [], []
        for i, x in enumerate(feats):
            c = self.cls_tower(x)
            r = self.reg_tower(x)

            cls_out.append(self.cls_logits(c))

            reg = self.scales[i](self.bbox_pred(r))
            reg = F.relu(reg).clamp(0.0, 256.0)
            reg_out.append(reg)

            ctr_out.append(self.ctr_logits(r))
        return cls_out, reg_out, ctr_out

In [ ]:
class MobileNetV3SmallBackbone(nn.Module):
    def __init__(self, use_pretrained: bool = True, use_stride4: bool = True, probe_size: int = 256):
        super().__init__()
        weights = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None
        m = torchvision.models.mobilenet_v3_small(weights=weights)
        self.features = m.features
        self.use_stride4 = use_stride4

        if self.use_stride4:
            return_nodes = {"1": "c2", "2": "c3", "7": "c4", "12": "c5"}
        else:
            return_nodes = {"2": "c3", "7": "c4", "12": "c5"}

        self.extractor = create_feature_extractor(self.features, return_nodes=return_nodes)

        with torch.no_grad():
            x = torch.zeros(1, 3, probe_size, probe_size)
            out = self.extractor(x)
            keys = ["c2", "c3", "c4", "c5"] if self.use_stride4 else ["c3", "c4", "c5"]
            self.out_channels = [int(out[k].shape[1]) for k in keys]

    def forward(self, x):
        out = self.extractor(x)
        if self.use_stride4:
            return [out["c2"], out["c3"], out["c4"], out["c5"]]
        return [out["c3"], out["c4"], out["c5"]]

In [ ]:
class OviDetLitePro(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.backbone = MobileNetV3SmallBackbone(use_pretrained=cfg.use_pretrained_backbone, use_stride4=cfg.use_stride4)
        self.neck = PANet4(
            in_channels=self.backbone.out_channels,
            fpn_dim=cfg.fpn_dim,
            use_ghost=cfg.use_ghost_neck,
            ghost_ratio=cfg.ghost_ratio,
            use_stride4=cfg.use_stride4,
            use_se=cfg.use_se_neck,
            use_cbam=cfg.use_cbam_neck,
            use_dilation=cfg.use_dilated_conv,
            dilation_rate=cfg.dilation_rate,
        )
        num_levels = 4 if cfg.use_stride4 else 3
        self.head = FCOSHead(
            fpn_dim=cfg.fpn_dim,
            num_classes=num_classes,
            num_levels=num_levels,
            depth=cfg.head_depth,
            use_depthwise=cfg.use_depthwise_head,
            use_dilation=cfg.use_dilated_conv,
            dilation_rate=cfg.dilation_rate,
        )

    def forward(self, x, return_feats: bool = False):
        feats = self.backbone(x)
        feats = self.neck(feats)
        out = self.head(feats)
        if return_feats:
            return out, feats
        return out

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

## Target assignment

Multi-level FCOS points, scale ranges, and positive-sample assignment.

In [ ]:
# Section 11 Points cache and target assignment

def make_points(h: int, w: int, stride: int, device: torch.device) -> torch.Tensor:
    ys = torch.arange(h, device=device) * stride + stride * 0.5
    xs = torch.arange(w, device=device) * stride + stride * 0.5
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([xx, yy], dim=-1).reshape(-1, 2)

def get_level_range_for_stride(strides: torch.Tensor, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    lo = torch.zeros_like(strides, dtype=torch.float32, device=device)
    hi = torch.zeros_like(strides, dtype=torch.float32, device=device)

    s4 = (strides == 4)
    s8 = (strides == 8)
    s16 = (strides == 16)
    s32 = (strides == 32)

    lo[s4] = cfg.range_s4[0]
    hi[s4] = cfg.range_s4[1]
    lo[s8] = cfg.range_s8[0]
    hi[s8] = cfg.range_s8[1]
    lo[s16] = cfg.range_s16[0]
    hi[s16] = cfg.range_s16[1]
    lo[s32] = cfg.range_s32[0]
    hi[s32] = cfg.range_s32[1]

    unk = ~(s4 | s8 | s16 | s32)
    lo[unk] = 0.0
    hi[unk] = 1e8
    return lo, hi

class PointsCache:
    def __init__(self):
        self.points = None
        self.strides = None
        self.level_slices = None

    @torch.no_grad()
    def build_from_model(self, model: nn.Module, device: torch.device, image_size: int):
        model.eval()
        x = torch.zeros(1, 3, image_size, image_size, device=device)
        cls_outs, reg_outs, ctr_outs = model(x)
        strides_list = [4, 8, 16, 32] if cfg.use_stride4 else [8, 16, 32]

        all_points = []
        all_strides = []
        level_slices = []
        start = 0
        for lvl, st in enumerate(strides_list):
            hh, ww = cls_outs[lvl].shape[-2], cls_outs[lvl].shape[-1]
            pts = make_points(hh, ww, st, device=device)
            all_points.append(pts)
            all_strides.append(torch.full((pts.shape[0],), st, device=device, dtype=torch.long))
            end = start + pts.shape[0]
            level_slices.append((start, end))
            start = end

        self.points = torch.cat(all_points, dim=0)
        self.strides = torch.cat(all_strides, dim=0)
        self.level_slices = level_slices
        return self.points, self.strides, self.level_slices

def build_targets_fast(points: torch.Tensor, strides: torch.Tensor, targets: Dict[str, torch.Tensor], num_classes: int, device: torch.device):
    boxes = targets["boxes"].to(device)
    labels = targets["labels"].to(device)

    P = points.shape[0]
    if boxes.numel() == 0:
        cls_t = torch.zeros((P, num_classes), device=device)
        reg_t = torch.zeros((P, 4), device=device)
        ctr_t = torch.zeros((P,), device=device)
        pos = torch.zeros((P,), device=device, dtype=torch.bool)
        return cls_t, reg_t, ctr_t, pos

    px = points[:, 0].unsqueeze(1)
    py = points[:, 1].unsqueeze(1)

    x1 = boxes[:, 0].unsqueeze(0)
    y1 = boxes[:, 1].unsqueeze(0)
    x2 = boxes[:, 2].unsqueeze(0)
    y2 = boxes[:, 3].unsqueeze(0)

    l = px - x1
    t = py - y1
    r = x2 - px
    b = y2 - py

    ltrb = torch.stack([l, t, r, b], dim=-1)  # P G 4
    inside = (ltrb.min(dim=-1).values > 0)

    max_ltrb = ltrb.max(dim=-1).values
    lo, hi = get_level_range_for_stride(strides.to(device), device=device)
    lo = lo.unsqueeze(1)
    hi = hi.unsqueeze(1)
    in_range = (max_ltrb >= lo) & (max_ltrb <= hi)

    valid = inside & in_range

    areas = ((boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)).unsqueeze(0)
    cand = areas.expand(P, -1).clone()
    cand[~valid] = float("inf")

    best_area, best_idx = cand.min(dim=1)
    pos = best_area < float("inf")

    assigned_labels = torch.zeros((P,), device=device, dtype=torch.long)
    assigned_ltrb = torch.zeros((P, 4), device=device)

    if pos.any():
        assigned_labels[pos] = labels[best_idx[pos]]
        assigned_ltrb[pos] = ltrb[pos, best_idx[pos]]

    cls_t = torch.zeros((P, num_classes), device=device)
    cls_t[pos, assigned_labels[pos]] = 1.0

    l2, t2, r2, b2 = assigned_ltrb.unbind(-1)
    ctr_t = torch.sqrt(
        (torch.minimum(l2, r2) / (torch.maximum(l2, r2) + 1e-6)) *
        (torch.minimum(t2, b2) / (torch.maximum(t2, b2) + 1e-6))
    ).clamp(0, 1)

    reg_t = assigned_ltrb
    return cls_t, reg_t, ctr_t, pos

## Post-processing

Box decoding and class-wise non-maximum suppression.

In [ ]:
# =========================================================
# Section 12 NMS
# =========================================================
def xyxy_from_points_ltrb(points_xy: torch.Tensor, ltrb: torch.Tensor) -> torch.Tensor:
    x, y = points_xy[:, 0], points_xy[:, 1]
    l, t, r, b = ltrb.unbind(-1)
    return torch.stack([x - l, y - t, x + r, y + b], dim=-1)

def nms_xyxy(boxes: torch.Tensor, scores: torch.Tensor, thr: float) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.long, device=boxes.device)
    x1, y1, x2, y2 = boxes.unbind(-1)
    areas = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0]
        keep.append(i)
        if order.numel() == 1:
            break
        rest = order[1:]
        xx1 = torch.maximum(x1[i], x1[rest])
        yy1 = torch.maximum(y1[i], y1[rest])
        xx2 = torch.minimum(x2[i], x2[rest])
        yy2 = torch.minimum(y2[i], y2[rest])
        w = (xx2 - xx1).clamp(min=0)
        h = (yy2 - yy1).clamp(min=0)
        inter = w * h
        iou = inter / (areas[i] + areas[rest] - inter + 1e-6)
        order = rest[iou <= thr]
    return torch.stack(keep)

## Training and evaluation

Optimization, COCO export/evaluation, checkpointing, and latency measurement.

In [ ]:
# =========================================================
# Section 13 Training and evaluation
# =========================================================
def build_optimizer(model: nn.Module):
    params_backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    params_rest = [p for n, p in model.named_parameters() if ("backbone" not in n) and p.requires_grad]
    opt = torch.optim.AdamW(
        [
            {"params": params_backbone, "lr": cfg.lr_backbone},
            {"params": params_rest, "lr": cfg.lr_head},
        ],
        weight_decay=cfg.weight_decay,
    )
    return opt

@torch.no_grad()
def coco_eval_stats(gt_ann_path: str, det_json_path: str):
    with open(det_json_path, "r", encoding="utf-8") as f:
        dets = json.load(f)
    if len(dets) == 0:
        print("No detections, evaluation skipped")
        return None
    coco_gt = COCO(gt_ann_path)
    coco_dt = coco_gt.loadRes(det_json_path)
    ev = COCOeval(coco_gt, coco_dt, iouType="bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    return ev.stats

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, scaler, device: torch.device, points_cache: PointsCache, class_alpha: Optional[torch.Tensor]):
    model.train()
    total_loss = 0.0

    points = points_cache.points
    strides = points_cache.strides
    level_slices = points_cache.level_slices

    for imgs, targets_list in loader:
        imgs = imgs.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        use_cuda_amp = (scaler is not None) and (device.type == "cuda")
        with torch.amp.autocast(device_type="cuda", enabled=use_cuda_amp):
            cls_outs, reg_outs, ctr_outs = model(imgs)

            B = imgs.shape[0]
            loss_cls = 0.0
            loss_reg = 0.0
            loss_ctr = 0.0

            # Flatten predictions per image
            for b in range(B):
                cls_list = []
                reg_list = []
                ctr_list = []

                for lvl, (s0, s1) in enumerate(level_slices):
                    c = cls_outs[lvl][b].permute(1, 2, 0).reshape(-1, cfg.num_classes)
                    r = reg_outs[lvl][b].permute(1, 2, 0).reshape(-1, 4)
                    t = ctr_outs[lvl][b].permute(1, 2, 0).reshape(-1)
                    cls_list.append(c)
                    reg_list.append(r)
                    ctr_list.append(t)

                cls_logits = torch.cat(cls_list, dim=0)
                reg_pred = torch.cat(reg_list, dim=0)
                ctr_logits = torch.cat(ctr_list, dim=0)

                cls_t, reg_t, ctr_t, pos = build_targets_fast(points, strides, targets_list[b], cfg.num_classes, device)
                num_pos = pos.sum().clamp(min=1).float()

                lc = sigmoid_focal_loss(cls_logits, cls_t, alpha_vec=class_alpha).sum() / num_pos
                loss_cls = loss_cls + lc

                if pos.any():
                    lr = iou_loss(reg_pred[pos], reg_t[pos]).sum() / num_pos
                    lt = F.binary_cross_entropy_with_logits(ctr_logits[pos], ctr_t[pos], reduction="sum") / num_pos
                    loss_reg = loss_reg + lr
                    loss_ctr = loss_ctr + lt

            loss = cfg.loss_weight_cls * loss_cls + cfg.loss_weight_reg * loss_reg + cfg.loss_weight_ctr * loss_ctr

        if scaler is None:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()

        total_loss += float(loss.detach().cpu().item())

    return total_loss / max(1, len(loader))

@torch.no_grad()
def run_inference_export(model: nn.Module, loader: DataLoader, device: torch.device, points_cache: PointsCache,
                         score_thr: float, nms_thr: float, max_det: int, contig_to_cat_id: Dict[int, int],
                         out_json_path: str):
    model.eval()
    all_dets = []

    points = points_cache.points
    level_slices = points_cache.level_slices

    use_cuda_amp = (device.type == "cuda") and cfg.amp
    for imgs, targets_list in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=use_cuda_amp):
            cls_outs, reg_outs, ctr_outs = model(imgs)

        B = imgs.shape[0]
        for b in range(B):
            image_id = int(targets_list[b]["image_id"])

            cls_list = []
            reg_list = []
            ctr_list = []
            for lvl, (s0, s1) in enumerate(level_slices):
                c = cls_outs[lvl][b].permute(1, 2, 0).reshape(-1, cfg.num_classes)
                r = reg_outs[lvl][b].permute(1, 2, 0).reshape(-1, 4)
                t = ctr_outs[lvl][b].permute(1, 2, 0).reshape(-1)
                cls_list.append(c)
                reg_list.append(r)
                ctr_list.append(t)

            cls_logits = torch.cat(cls_list, dim=0)
            reg_pred = torch.cat(reg_list, dim=0)
            ctr_logits = torch.cat(ctr_list, dim=0)

            cls_prob = torch.sigmoid(cls_logits)
            ctr_prob = torch.sigmoid(ctr_logits)
            scores, labels = cls_prob.max(dim=-1)
            scores = scores * ctr_prob

            keep = scores > score_thr
            if keep.sum() == 0 and cfg.eval_force_topk > 0:
                k = min(cfg.eval_force_topk, scores.numel())
                topk = torch.topk(scores, k=k, largest=True).indices
                keep = torch.zeros_like(scores, dtype=torch.bool)
                keep[topk] = True

            if keep.sum() == 0:
                continue

            pts_k = points[keep]
            reg_k = reg_pred[keep]
            scores_k = scores[keep]
            labels_k = labels[keep]

            boxes = xyxy_from_points_ltrb(pts_k, reg_k)

            keep_all = []
            for c in range(cfg.num_classes):
                idx = torch.where(labels_k == c)[0]
                if idx.numel() == 0:
                    continue
                kidx = nms_xyxy(boxes[idx], scores_k[idx], nms_thr)
                keep_all.append(idx[kidx])
            if len(keep_all) == 0:
                continue

            keep_idx = torch.cat(keep_all, dim=0)
            keep_idx = keep_idx[scores_k[keep_idx].argsort(descending=True)][:max_det]

            boxes_np = boxes[keep_idx].detach().cpu().numpy()
            scores_np = scores_k[keep_idx].detach().cpu().numpy()
            labels_np = labels_k[keep_idx].detach().cpu().numpy()

            # Undo letterbox for val and test
            meta = targets_list[b].get("meta", None)
            if meta is not None:
                s = float(meta["scale"])
                left = float(meta["pad_left"])
                top = float(meta["pad_top"])
                oh = float(meta["orig_h"])
                ow = float(meta["orig_w"])

                boxes_np[:, 0] = (boxes_np[:, 0] - left) / s
                boxes_np[:, 1] = (boxes_np[:, 1] - top) / s
                boxes_np[:, 2] = (boxes_np[:, 2] - left) / s
                boxes_np[:, 3] = (boxes_np[:, 3] - top) / s
                boxes_np[:, 0] = np.clip(boxes_np[:, 0], 0, ow - 1)
                boxes_np[:, 2] = np.clip(boxes_np[:, 2], 0, ow - 1)
                boxes_np[:, 1] = np.clip(boxes_np[:, 1], 0, oh - 1)
                boxes_np[:, 3] = np.clip(boxes_np[:, 3], 0, oh - 1)

            for box, sc, lab in zip(boxes_np, scores_np, labels_np):
                cat_id = int(contig_to_cat_id[int(lab)])
                all_dets.append({
                    "image_id": image_id,
                    "category_id": cat_id,
                    "bbox": xywh_from_xyxy(box),
                    "score": float(sc),
                })

    ensure_dir(os.path.dirname(out_json_path))
    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(all_dets, f)
    return out_json_path

@torch.no_grad()
def benchmark(model: nn.Module, device: torch.device, image_size: int, iters: int = 200, warmup: int = 50):
    model.eval()
    x = torch.randn(1, 3, image_size, image_size, device=device)
    for _ in range(warmup):
        _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(iters):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.time()

    avg = (t1 - t0) / iters
    fps = 1.0 / max(1e-9, avg)
    return avg, fps

def save_checkpoint(model: nn.Module, path: str, epoch: int, val_map: Optional[float] = None, fp16: bool = False):
    state = model.state_dict()
    if fp16:
        state = {k: v.detach().half().cpu() for k, v in state.items()}
    payload = {"model": state, "epoch": int(epoch)}
    if val_map is not None:
        payload["val_mAP"] = float(val_map)
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)

def load_checkpoint(model: nn.Module, path: str, device: torch.device):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)
    model.to(device)
    return ckpt


## Inference engine

Single-image prediction with letterbox reversal and visualization.

In [ ]:
# =========================================================
# Section 14 Inference engine only, industry style
# =========================================================
class InferenceEngine:
    def __init__(self, checkpoint_path: str, cat_map_contig_to_id: Dict[int, int], device_str: str = "cuda"):
        self.device = torch.device(device_str if torch.cuda.is_available() else "cpu")
        self.model = OviDetLitePro(num_classes=cfg.num_classes).to(self.device)
        self.model.eval()
        _ = load_checkpoint(self.model, checkpoint_path, self.device)

        self.points_cache = PointsCache()
        self.points_cache.build_from_model(self.model, self.device, cfg.image_size)

        self.contig_to_cat_id = cat_map_contig_to_id

    @torch.no_grad()
    def predict_single_image(self, img_bgr: np.ndarray, score_thr: float, nms_thr: float, max_det: int):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_lb, _, meta = letterbox_resize(img_rgb, None, cfg.image_size)
        x = torch.from_numpy(img_lb).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        x = x.to(self.device)

        use_cuda_amp = (self.device.type == "cuda") and cfg.amp
        with torch.amp.autocast(device_type="cuda", enabled=use_cuda_amp):
            cls_outs, reg_outs, ctr_outs = self.model(x)

        points = self.points_cache.points
        level_slices = self.points_cache.level_slices

        cls_list = []
        reg_list = []
        ctr_list = []
        for lvl, (s0, s1) in enumerate(level_slices):
            c = cls_outs[lvl][0].permute(1, 2, 0).reshape(-1, cfg.num_classes)
            r = reg_outs[lvl][0].permute(1, 2, 0).reshape(-1, 4)
            t = ctr_outs[lvl][0].permute(1, 2, 0).reshape(-1)
            cls_list.append(c)
            reg_list.append(r)
            ctr_list.append(t)

        cls_logits = torch.cat(cls_list, dim=0)
        reg_pred = torch.cat(reg_list, dim=0)
        ctr_logits = torch.cat(ctr_list, dim=0)

        cls_prob = torch.sigmoid(cls_logits)
        ctr_prob = torch.sigmoid(ctr_logits)
        scores, labels = cls_prob.max(dim=-1)
        scores = scores * ctr_prob

        keep = scores > score_thr
        if keep.sum() == 0 and cfg.eval_force_topk > 0:
            k = min(cfg.eval_force_topk, scores.numel())
            topk = torch.topk(scores, k=k, largest=True).indices
            keep = torch.zeros_like(scores, dtype=torch.bool)
            keep[topk] = True

        if keep.sum() == 0:
            return [], meta

        pts_k = points[keep]
        reg_k = reg_pred[keep]
        scores_k = scores[keep]
        labels_k = labels[keep]

        boxes = xyxy_from_points_ltrb(pts_k, reg_k)

        keep_all = []
        for c in range(cfg.num_classes):
            idx = torch.where(labels_k == c)[0]
            if idx.numel() == 0:
                continue
            kidx = nms_xyxy(boxes[idx], scores_k[idx], nms_thr)
            keep_all.append(idx[kidx])
        if len(keep_all) == 0:
            return [], meta

        keep_idx = torch.cat(keep_all, dim=0)
        keep_idx = keep_idx[scores_k[keep_idx].argsort(descending=True)][:max_det]

        boxes_np = boxes[keep_idx].detach().cpu().numpy()
        scores_np = scores_k[keep_idx].detach().cpu().numpy()
        labels_np = labels_k[keep_idx].detach().cpu().numpy()

        # Undo letterbox to original image size
        s = float(meta["scale"])
        left = float(meta["pad_left"])
        top = float(meta["pad_top"])
        oh = float(meta["orig_h"])
        ow = float(meta["orig_w"])

        boxes_np[:, 0] = (boxes_np[:, 0] - left) / s
        boxes_np[:, 1] = (boxes_np[:, 1] - top) / s
        boxes_np[:, 2] = (boxes_np[:, 2] - left) / s
        boxes_np[:, 3] = (boxes_np[:, 3] - top) / s
        boxes_np[:, 0] = np.clip(boxes_np[:, 0], 0, ow - 1)
        boxes_np[:, 2] = np.clip(boxes_np[:, 2], 0, ow - 1)
        boxes_np[:, 1] = np.clip(boxes_np[:, 1], 0, oh - 1)
        boxes_np[:, 3] = np.clip(boxes_np[:, 3], 0, oh - 1)

        dets = []
        for box, sc, lab in zip(boxes_np, scores_np, labels_np):
            dets.append({
                "category_id": int(self.contig_to_cat_id[int(lab)]),
                "score": float(sc),
                "bbox_xyxy": [float(v) for v in box.tolist()],
                "bbox_xywh": xywh_from_xyxy(box),
            })
        return dets, meta

    def visualize_single_image(self, img_bgr: np.ndarray, dets: List[Dict[str, Any]], id_to_name: Dict[int, str]):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        boxes = [d["bbox_xyxy"] for d in dets]
        labels = [id_to_name.get(int(d["category_id"]), str(d["category_id"])) for d in dets]
        scores = [float(d["score"]) for d in dets]
        vis = draw_boxes(img_rgb, boxes, labels, scores=scores, color=(0, 255, 0))
        return vis

## Optional ONNX export

In [ ]:
# Section 15 Optional ONNX export
# =========================================================
def export_onnx(model: nn.Module, out_path: str, image_size: int, device: torch.device):
    ensure_dir(os.path.dirname(out_path))
    model.eval()
    x = torch.zeros(1, 3, image_size, image_size, device=device)
    with torch.no_grad():
        torch.onnx.export(
            model,
            x,
            out_path,
            opset_version=17,
            input_names=["images"],
            output_names=["cls_outs", "reg_outs", "ctr_outs"],
            do_constant_folding=True,
        )
    return out_path

## End-to-end experiment

Audit, train, select by validation mAP, evaluate on test, benchmark, and save artifacts.

In [ ]:
# =========================================================
# Section 16 Main pipeline
# =========================================================
def main():
    set_seed(cfg.seed)
    ensure_dir(cfg.out_dir)

    # Speed setup
    try:
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

    device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    print("Model:", cfg.model_name)

    ann_train_path = os.path.join(cfg.data_root, cfg.ann_train)
    ann_val_path = os.path.join(cfg.data_root, cfg.ann_val)
    ann_test_path = os.path.join(cfg.data_root, cfg.ann_test)

    coco_train = load_coco_json(ann_train_path)

    # Map class name to category id in COCO json
    categories = sorted(coco_train["categories"], key=lambda x: x["id"])
    cat_name_to_id = {c["name"]: int(c["id"]) for c in categories}

    # Build category id maps for selected classes
    cat_id_to_contig: Dict[int, int] = {}
    contig_to_cat_id: Dict[int, int] = {}

    for i, name in enumerate(cfg.class_names):
        if name not in cat_name_to_id:
            raise ValueError(f"Category name not found in annotations: {name}")
        cid = int(cat_name_to_id[name])
        cat_id_to_contig[cid] = int(i)
        contig_to_cat_id[int(i)] = cid

    keep_cat_ids = set(cat_id_to_contig.keys())

    tr_dir = os.path.join(cfg.data_root, cfg.train_images)
    va_dir = os.path.join(cfg.data_root, cfg.val_images)
    te_dir = os.path.join(cfg.data_root, cfg.test_images)

    # Step A Data audit and visuals
    audit_dir = os.path.join(cfg.out_dir, "audit")
    ensure_dir(audit_dir)

    print("\nStep A audit train")
    tr_a = audit_coco_split(ann_train_path, tr_dir, keep_cat_ids)
    print(tr_a)
    save_audit_reports(tr_a, audit_dir, "train")
    overlay_random_gt_samples(tr_dir, ann_train_path, os.path.join(audit_dir, "gt_train_samples"), keep_cat_ids, cfg.audit_num_overlay)
    overlay_bad_box_samples(tr_dir, ann_train_path, os.path.join(audit_dir, "bad_train_samples"), keep_cat_ids, cfg.audit_num_badbox_overlay)

    print("\nStep A audit val")
    va_a = audit_coco_split(ann_val_path, va_dir, keep_cat_ids)
    print(va_a)
    save_audit_reports(va_a, audit_dir, "val")
    overlay_random_gt_samples(va_dir, ann_val_path, os.path.join(audit_dir, "gt_val_samples"), keep_cat_ids, cfg.audit_num_overlay)

    print("\nStep A audit test")
    te_a = audit_coco_split(ann_test_path, te_dir, keep_cat_ids)
    print(te_a)
    save_audit_reports(te_a, audit_dir, "test")
    overlay_random_gt_samples(te_dir, ann_test_path, os.path.join(audit_dir, "gt_test_samples"), keep_cat_ids, cfg.audit_num_overlay)

    # Step B Datasets and loaders
    print("\nStep B build datasets and loaders")
    train_ds = CocoDetDataset(tr_dir, ann_train_path, cfg.image_size, True, cat_id_to_contig)
    val_ds = CocoDetDataset(va_dir, ann_val_path, cfg.image_size, False, cat_id_to_contig)
    test_ds = CocoDetDataset(te_dir, ann_test_path, cfg.image_size, False, cat_id_to_contig)

    train_ld = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
        collate_fn=collate_fn,
    )
    val_ld = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
        collate_fn=collate_fn,
    )
    test_ld = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
        collate_fn=collate_fn,
    )

    # Step C Class alpha for focal loss
    print("\nStep C class alpha from train annotations")
    counts = np.zeros((cfg.num_classes,), dtype=np.float64)
    for ann in coco_train["annotations"]:
        cid = int(ann["category_id"])
        if cid in cat_id_to_contig:
            counts[cat_id_to_contig[cid]] += 1.0
    counts = np.maximum(counts, 1.0)
    inv = 1.0 / counts
    inv = inv / inv.sum()
    class_alpha = torch.tensor(inv, dtype=torch.float32)
    print("Class counts:", counts.tolist())
    print("Class alpha:", class_alpha.tolist())

    # Step D Build model
    print("\nStep D build model")
    model = OviDetLitePro(num_classes=cfg.num_classes).to(device)
    print("Backbone out channels:", model.backbone.out_channels)
    print("fpn_dim:", cfg.fpn_dim, "head_depth:", cfg.head_depth, "depthwise_head:", cfg.use_depthwise_head)

    if cfg.use_pretrained_backbone and cfg.freeze_backbone_epochs > 0:
        model.freeze_backbone()
        print("Backbone frozen for epochs:", cfg.freeze_backbone_epochs)

    if cfg.compile_model and hasattr(torch, "compile") and device.type == "cuda":
        model = torch.compile(model)
        print("torch compile enabled")

    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type == "cuda") else None

    # Step E Build points cache once, this reduces first epoch overhead
    points_cache = PointsCache()
    points_cache.build_from_model(model, device, cfg.image_size)

    # Warmup forward to reduce first epoch latency on some setups
    with torch.no_grad():
        xw = torch.zeros(1, 3, cfg.image_size, cfg.image_size, device=device)
        _ = model(xw)
        if device.type == "cuda":
            torch.cuda.synchronize()

    best_map = -1.0
    best_path = os.path.join(cfg.out_dir, "best.pt")
    best_fp16_path = os.path.join(cfg.out_dir, "best_fp16.pt")
    last_path = os.path.join(cfg.out_dir, "last.pt")
    last_fp16_path = os.path.join(cfg.out_dir, "last_fp16.pt")

    train_loss_hist = []
    val_map_hist = []
    lr_hist = []
    thr_hist = []

    # Step F Training loop
    print("\nStep F training")
    for epoch in range(cfg.epochs):
        epoch_1 = epoch + 1

        if cfg.use_pretrained_backbone and cfg.freeze_backbone_epochs > 0:
            if epoch_1 == cfg.freeze_backbone_epochs + 1:
                model.unfreeze_backbone()
                optimizer = build_optimizer(model)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(cfg.epochs - epoch), eta_min=1e-6)
                print("Backbone unfrozen at epoch:", epoch_1)

        eval_thr = get_eval_score_thresh(epoch_1)
        thr_hist.append(eval_thr)

        lr_hist.append(float(optimizer.param_groups[1]["lr"]))

        loss = train_one_epoch(model, train_ld, optimizer, scaler, device, points_cache, class_alpha)
        scheduler.step()
        train_loss_hist.append(loss)

        print(f"Epoch {epoch_1:03d} train_loss {loss:.6f} lr_head {lr_hist[-1]:.7f} eval_thr {eval_thr}")

        save_checkpoint(model, last_path, epoch_1, val_map=None, fp16=False)
        if cfg.save_fp16_checkpoint:
            save_checkpoint(model, last_fp16_path, epoch_1, val_map=None, fp16=True)

        if epoch_1 % cfg.eval_every == 0:
            det_val_path = os.path.join(cfg.out_dir, f"detections_val_epoch_{epoch_1:03d}.json")
            run_inference_export(
                model=model,
                loader=val_ld,
                device=device,
                points_cache=points_cache,
                score_thr=eval_thr,
                nms_thr=cfg.nms_thresh,
                max_det=cfg.max_det,
                contig_to_cat_id=contig_to_cat_id,
                out_json_path=det_val_path,
            )
            stats = coco_eval_stats(ann_val_path, det_val_path)
            cur_map = float(stats[0]) if stats is not None else 0.0
            val_map_hist.append(cur_map)
            print(f"Epoch {epoch_1:03d} val_mAP {cur_map:.6f}")

            if cur_map > best_map:
                best_map = cur_map
                save_checkpoint(model, best_path, epoch_1, val_map=best_map, fp16=False)
                if cfg.save_fp16_checkpoint:
                    save_checkpoint(model, best_fp16_path, epoch_1, val_map=best_map, fp16=True)
                print("Saved best checkpoint")

    # Step G Test evaluation with best checkpoint
    print("\nStep G test evaluation using best checkpoint")
    _ = load_checkpoint(model, best_path, device)
    points_cache.build_from_model(model, device, cfg.image_size)

    det_test_path = os.path.join(cfg.out_dir, "detections_test.json")
    run_inference_export(
        model=model,
        loader=test_ld,
        device=device,
        points_cache=points_cache,
        score_thr=get_eval_score_thresh(cfg.epochs),
        nms_thr=cfg.nms_thresh,
        max_det=cfg.max_det,
        contig_to_cat_id=contig_to_cat_id,
        out_json_path=det_test_path,
    )

    print("\nCOCO evaluation on test split")
    _ = coco_eval_stats(ann_test_path, det_test_path)

    # Step H Speed benchmark
    print("\nStep H speed benchmark")
    avg, fps = benchmark(model, device, cfg.image_size, iters=200, warmup=50)
    print(f"Latency seconds {avg:.6f} FPS {fps:.2f}")

    # Step I Curves
    print("\nStep I save curves")
    try:
        import matplotlib.pyplot as plt
        curves_dir = os.path.join(cfg.out_dir, "curves")
        ensure_dir(curves_dir)

        plt.figure()
        plt.plot(train_loss_hist)
        plt.xlabel("epoch")
        plt.ylabel("train_loss")
        plt.title("Training loss")
        plt.savefig(os.path.join(curves_dir, "train_loss.png"), dpi=160, bbox_inches="tight")
        plt.close()

        plt.figure()
        plt.plot(val_map_hist)
        plt.xlabel("eval_step")
        plt.ylabel("val_mAP")
        plt.title("Validation mAP")
        plt.savefig(os.path.join(curves_dir, "val_map.png"), dpi=160, bbox_inches="tight")
        plt.close()

        plt.figure()
        plt.plot(thr_hist)
        plt.xlabel("epoch")
        plt.ylabel("eval_score_thr")
        plt.title("Eval threshold schedule")
        plt.savefig(os.path.join(curves_dir, "eval_thr.png"), dpi=160, bbox_inches="tight")
        plt.close()

        plt.figure()
        plt.plot(lr_hist)
        plt.xlabel("epoch")
        plt.ylabel("lr_head")
        plt.title("Learning rate head")
        plt.savefig(os.path.join(curves_dir, "lr_head.png"), dpi=160, bbox_inches="tight")
        plt.close()

    except Exception as e:
        print("Curve plotting skipped:", e)

    # Step J Optional ONNX export
    if cfg.export_onnx and device.type in ["cuda", "cpu"]:
        print("\nStep J export ONNX")
        path = export_onnx(model, cfg.onnx_path, cfg.image_size, device)
        print("Saved ONNX:", path)

    # Step K Save run metrics
    metrics_path = os.path.join(cfg.out_dir, "run_metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "model_name": cfg.model_name,
                "best_val_map": float(best_map),
                "train_loss_hist": train_loss_hist,
                "val_map_hist": val_map_hist,
                "thr_hist": thr_hist,
                "lr_hist": lr_hist,
                "fps": float(fps),
                "latency_seconds": float(avg),
                "cfg": cfg.__dict__,
            },
            f,
            indent=2,
        )

    # Step L Print artifact paths
    print("\nSaved outputs in:", cfg.out_dir)
    print("Best checkpoint:", best_path)
    print("Best fp16 checkpoint:", best_fp16_path if cfg.save_fp16_checkpoint else "disabled")
    print("Last checkpoint:", last_path)
    print("Last fp16 checkpoint:", last_fp16_path if cfg.save_fp16_checkpoint else "disabled")
    print("Test detections json:", det_test_path)
    print("Run metrics json:", metrics_path)
    print("Audit folder:", audit_dir)

    # Size hint
    if cfg.save_fp16_checkpoint and os.path.exists(best_fp16_path):
        try:
            sz = os.path.getsize(best_fp16_path) / (1024 * 1024)
            print("Best fp16 size MB:", round(sz, 3))
        except Exception:
            pass


if __name__ == "__main__":
    main()